In [5]:
import py4cytoscape as p4c
p4c.cytoscape_ping()
p4c.cytoscape_version_info()

You are connected to Cytoscape!


{'apiVersion': 'v1',
 'cytoscapeVersion': '3.10.3',
 'automationAPIVersion': '1.11.0',
 'py4cytoscapeVersion': '1.11.0'}

In [6]:
import re
import math

In [7]:
node_table = p4c.get_table_columns('node')

# Copy label to name (for example)
node_table["name"] = node_table['label']
for index, row in node_table.iterrows():
    if row["sbml type ext"] == "eventAssignment":
        label = row["label"]
        if label.startswith("event_p_"):
            math_value = row["math"]
            new_name = f"ea_{label}_{math_value}"
        else:
            new_name = f"ea_{label}"
        node_table.loc[index, "name"] = new_name

# Write back to Cytoscape
p4c.load_table_data(node_table, data_key_column="SUID", table_key_column="SUID")

'Success: Data loaded in defaultnode table'

In [8]:
event_re = re.compile(r"^event_\d+$")
number_re = re.compile(r"\d+")
rnn_neuron_re = re.compile(r"^p_n_\d+_\d+$")

names = p4c.networks.get_all_nodes()
weights = [node for node in names if node.startswith("w_") or node.startswith("b_")]
p4c.hide_nodes(weights + ["m2", "METRE", "mmol", "MOLE", "litre_base", "LITRE"])
events = [node for node in names if re.match(event_re, node)]
def sorting_fun(s):
    num = re.findall(number_re, s)[0]
    return int(num)
events.sort(key=sorting_fun)
'''xs = [x*500 for x in range(len(events))]
ys = [1000] * len(events)'''
#print(events)
xs = []
ys = []

rnn_neurons = [node for node in names if re.match(rnn_neuron_re, node)]
rnn_neurons_layers = {}
for neuron in rnn_neurons:
    layer_num = int(re.findall(number_re, neuron)[0])
    if layer_num in rnn_neurons_layers.keys():
        rnn_neurons_layers[layer_num].append(neuron)
    else:
        rnn_neurons_layers[layer_num] = [neuron]
sorted_keys = sorted(rnn_neurons_layers)
def sorting_fun2(s):
    num = re.findall(number_re, s)[1]
    return int(num)
sorted_dict = {key: rnn_neurons_layers[key] for key in sorted_keys}
for val in sorted_dict.values():
    val.sort(key = sorting_fun2)

x0 = 0
y0 = 0
dx = 200
dy = 30
d_ea = 40

x0_past = x0 -200*len(sorted_dict.keys())
y0_past = 300

#to_pos = events[:]
to_pos = []
events_to_interm = []

for key, val in sorted_dict.items():
    for i, name in enumerate(val):
        to_pos.append(name)
        xs.append(x0 + key*dx)
        ys.append(y0 + i*dy)
        to_pos.append(f"ea_{name}")
        xs.append(x0 + key*dx - d_ea)
        ys.append(y0 + i*dy)
        past_name = f"past_{name[4:]}"
        to_pos.append(past_name)
        xs.append(x0_past + key*dx)
        ys.append(y0_past + i*dy)
        to_pos.append(f"ea_{past_name}")
        xs.append(x0_past + key*dx + d_ea)
        ys.append(y0_past + i*dy)
    to_pos.append(events[key])
    xs.append(x0 + key*dx - d_ea)
    ys.append(y0_past + dy*len(val))
    if key == 0:
        to_pos.append("ea_t_event")
        xs.append(x0 - d_ea)
        ys.append(y0_past + dy*len(val) + dy*2)
        to_pos.append("t_event")
        xs.append(x0 - d_ea - 1.8*dy)
        ys.append(y0_past + dy*len(val) + dy)
        to_pos.append("delay_event")
        xs.append(x0 - d_ea)
        ys.append(y0_past + dy*len(val) + dy*4)

    events_to_interm.append((xs[-1], ys[-1]))

inputs = ["t_scaled", "Feed0", "X0", "S0", "V0"]
time_additions = ["<t_scaled>", "t_max"]
x0_input = -200
y0_input = -200
for i, name in enumerate(inputs):
    if name == "t_scaled":
        for j, addit in enumerate(time_additions):
            to_pos.append(addit)
            xs.append(x0_input - (j+1)*d_ea)
            ys.append(y0_input)
        to_pos.append(name)
        xs.append(x0_input)
        ys.append(y0_input + i*dy*3)
    else:
        to_pos.append(f"{name}_scaled")
        xs.append(x0_input)
        ys.append(y0_input + i*dy*3)
        to_pos.append(f"assignment__{name}_scaled")
        xs.append(x0_input - d_ea)
        ys.append(y0_input + i*dy*3)
        to_pos.append(f"{name}_max")
        xs.append(x0_input - d_ea)
        ys.append(y0_input + i*dy*3 - dy*0.7)
        to_pos.append(f"{name}_min")
        xs.append(x0_input - d_ea)
        ys.append(y0_input + i*dy*3 + dy*0.7)
        to_pos.append(name)
        xs.append(x0_input - 2*d_ea)
        ys.append(y0_input + i*dy*3)


output_labels = ["X", "S", "PT", "PM", "V"]
x0_output = x0 + 200*len(sorted_dict.keys())
y0_output = y0 + ((len(sorted_dict[list(sorted_dict.keys())[-1]]) - 1) - (len(output_labels) - 1) * 3) * dy /2
dy_output = dy*3
for i, name in enumerate(output_labels):
    ea_name = f"ea_p_{name}"
    to_pos.append(ea_name)
    xs.append(x0_output)
    ys.append(y0_output + dy_output*i)
    p_name = f"p_{name}"
    to_pos.append(p_name)
    xs.append(x0_output + d_ea)
    ys.append(y0_output + dy_output*i)
    ar_name = f"<{name}>"
    to_pos.append(ar_name)
    xs.append(x0_output + d_ea*2)
    ys.append(y0_output + dy_output*i)
    min_name = f"{name}_min"
    to_pos.append(min_name)
    xs.append(x0_output + d_ea*2)
    ys.append(y0_output + dy_output*i + dy*0.7)
    max_name = f"{name}_max"
    to_pos.append(max_name)
    xs.append(x0_output + d_ea*2)
    ys.append(y0_output + dy_output*i - dy*0.7)
    to_pos.append(name)
    xs.append(x0_output + d_ea*3)
    ys.append(y0_output + dy_output*i)

c_seq = ["c", "litre_base", "LITRE"]
c_x = x0_output + d_ea * 5
c_y = y0 + (len(sorted_dict[list(sorted_dict.keys())[-1]]) - 1) * dy /2
for i, name in enumerate(c_seq):
    to_pos.append(name)
    xs.append(c_x + i*d_ea*2)
    ys.append(c_y)

ev_name = events[-1]
ev_x = x0_past + (len(sorted_dict.keys()) - 1) * dx / 2 + d_ea
ev_y = y0_past + dy*len(sorted_dict[list(sorted_dict.keys())[-1]]) + 200
to_pos.append(ev_name)
xs.append(ev_x)
ys.append(ev_y)

to_pos.append(events[-2])
xs.append(x0_output)
ys.append(ev_y)
events_to_interm.append((x0_output, ev_y))
events_to_interm.append((ev_x, ev_y))

def pose_interm_events(event_num, x1, x2, y1, y2, to_pos, xs, ys):
    to_pos.append(f"event_p_{event_num}")
    xs.append((x1 + x2)/2)
    ys.append((y1 + y2)/2)
    angle = math.atan2(y2-y1, x2-x1) - math.pi/4
    length = math.sqrt(2) * math.sqrt((x2 - x1)**2 + (y2-y1)**2) / 4
    x_len = math.cos(angle) * length
    y_len = math.sin(angle) * length
    to_pos.append(f"ea_event_p_{event_num}_1")
    xs.append(x1 + x_len)
    ys.append(y1 + y_len)
    to_pos.append(f"ea_event_p_{event_num}_0")
    xs.append((x1+x2)/2 + x_len)
    ys.append((y1+y2)/2 + y_len)

for i in range(len(events_to_interm) - 1):
    x1, y1 = events_to_interm[i]
    x2, y2 = events_to_interm[i+1]
    pose_interm_events(i, x1, x2, y1, y2, to_pos, xs, ys)

p4c.set_node_position_bypass(to_pos, new_x_locations=xs, new_y_locations=ys)

''